# Llama-3-Taiwan-8B — Hakka Medical Dialogue (四縣腔)

Fine-tune `yentinglin/Llama-3-Taiwan-8B-Instruct` để làm **bác sĩ nói tiếng Hakka (四縣腔)** trong domain y tế.

| Stage | Mô tả |
|-------|-------|
| **1. Data** | `客語漢字QA/` (41 dialogues) + `HAKKA_UTF8_TSV/` (Hakka side, 12 dialogues — dedup) |
| **2. Format** | Sliding window = 6 turns → predict Doctor turn |
| **3. Fine-tune** | LoRA (Unsloth), Doctor role only |
| **4. Evaluation** | BLEU · ROUGE-1/2/L · Perplexity — baseline vs fine-tuned |

**Dataset structure:**
- `客語漢字QA/` — 41 dialogues, 888 turns, Hakka-only (四縣 suy đoán)
- `HAKKA_UTF8_TSV/` — 12 dialogues (subset của QA), có thêm Hakka拼音 & Mandarin
- Sau dedup: **41 unique dialogues**
- Split: 5 dialogues (test) / 36 dialogues (train)

## 1. Install

In [2]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import re
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(__import__('torch').__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install -q sentencepiece protobuf "datasets>=2.19" "huggingface_hub>=0.34.0"
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q transformers==4.56.2 rouge-score

## 2. Configuration

In [1]:
import os, random

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_ROOT   = "../dataset"          # Colab
# DATASET_ROOT = "../../dataset"       # Local
QA_DIR  = os.path.join(DATASET_ROOT, "客語漢字QA")
TSV_DIR = os.path.join(DATASET_ROOT, "HAKKA_UTF8_TSV")

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID       = "yentinglin/Llama-3-Taiwan-8B-Instruct"
OUTPUT_DIR     = "outputs/hakka_dialogue"
LOAD_IN_4BIT   = True

# ── Dialogue config ───────────────────────────────────────────────────────────
WINDOW_SIZE    = 6      # turns of context before predicting Doctor turn
TEST_N         = 5      # number of dialogues held out for test (1 per topic if possible)
RANDOM_SEED    = 42

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "你係一位客語（四縣腔）醫生，請用純客語（四縣腔）同病人進行醫療問診。"
    "毋好用華語抑係英文回覆，所有回覆愛用客語漢字。"
)

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0      # Must be 0 for Unsloth fast patching (dropout=0.05 disables QKV/O/MLP optimization)
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj",
                  "gate_proj","up_proj","down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 512
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
MAX_STEPS      = 300    # 100 was not enough (loss stuck at 0.93 after 53 epochs with noisy data)
LR             = 2e-4
WARMUP_STEPS   = 5

# ── Eval ──────────────────────────────────────────────────────────────────────
EVAL_BATCH_SIZE = 4
MAX_NEW_TOKENS  = 128

random.seed(RANDOM_SEED)
print("Config OK")
print(f"Window size  : {WINDOW_SIZE} turns")
print(f"Test set     : {TEST_N} dialogues (held out)")
print(f"4-bit        : {LOAD_IN_4BIT}")
print(f"Data root    : {DATASET_ROOT}")
print(f"LoRA dropout : {LORA_DROPOUT}  (0 = Unsloth fast path enabled)")
print(f"Max steps    : {MAX_STEPS}")

Config OK
Window size  : 6 turns
Test set     : 5 dialogues (held out)
4-bit        : True
Data root    : ../dataset
LoRA dropout : 0  (0 = Unsloth fast path enabled)
Max steps    : 300


## 3. Data Loading

In [2]:
import glob

def load_qa_dialogues(qa_dir):
    """
    Load 客語漢字QA — mỗi file là 1 dialogue.
    Format: row 0 = Doctor, row 1 = Patient, strict alternation (per dataset design).
    Data issue: đôi khi Doctor nói 2+ dòng liên tiếp → shift role.
    Fix: detect shift bằng keyword markers và merge dòng lệch vào turn đúng.
    Returns list of dicts: {id, topic, source, turns: [(role, text), ...]}
    """
    # Keyword để xác nhận role tuyệt đối
    DOCTOR_KW  = ('恁樣哦', '𫣆俚', '𫣆先', '𠊎兜', '毋使恁細義', '祝你早兜仔', '𠊎建議', '𠊎知哩。')
    PATIENT_KW = ('先生，𠊎', '先生你好，𠊎', '承蒙先生，恁仔細',
                  '好哦，承蒙先生', '好，承蒙先生', '麻煩先生')

    dialogues = []
    for f in sorted(glob.glob(os.path.join(qa_dir, "*.tsv"))):
        with open(f, encoding='utf-8') as fp:
            lines = [l.strip() for l in fp if l.strip() and l.strip() != '客語漢字']
        if not lines:
            continue

        # Step 1: even/odd assignment (format gốc: row 0 = doctor)
        raw = [('doctor' if i % 2 == 0 else 'patient', line)
               for i, line in enumerate(lines)]

        # Step 2: fix confirmed-wrong lines rồi merge thành turns
        # Nếu line được label 'patient' nhưng có DOCTOR_KW → sai, merge vào doctor turn trước
        # Nếu line được label 'doctor' nhưng có PATIENT_KW → sai, merge vào patient turn trước
        turns = []
        for assigned_role, line in raw:
            confirmed_doctor  = any(kw in line for kw in DOCTOR_KW)
            confirmed_patient = any(kw in line for kw in PATIENT_KW)

            if confirmed_doctor and assigned_role == 'patient':
                actual_role = 'doctor'    # label sai — bác sĩ nói thêm dòng
            elif confirmed_patient and assigned_role == 'doctor':
                actual_role = 'patient'   # label sai — bệnh nhân nói thêm dòng
            else:
                actual_role = assigned_role  # giữ nguyên even/odd

            # Merge vào turn trước nếu cùng role, ngược lại tạo turn mới
            if turns and turns[-1][0] == actual_role:
                turns[-1] = (actual_role, turns[-1][1] + ' ' + line)
            else:
                turns.append((actual_role, line))

        name  = os.path.basename(f).replace('.tsv', '')
        topic = name.rstrip('0123456789')
        dialogues.append({
            'id'    : name,
            'topic' : topic,
            'source': 'QA',
            'turns' : turns,
        })
    return dialogues


def load_tsv_dialogues(tsv_dir):
    """
    Load HAKKA_UTF8_TSV — UTF-16-LE, 6 columns, explicit speaker labels.
    speaker01 = doctor | speaker02 = patient.
    Consecutive same-speaker lines are merged into one turn.
    Returns same format as load_qa_dialogues.
    """
    dialogues = []
    for f in sorted(glob.glob(os.path.join(tsv_dir, "*.txt"))):
        with open(f, encoding='utf-16-le') as fp:
            content = fp.read()
        lines = [l for l in content.splitlines() if l.strip()]
        raw = []
        for line in lines:
            cols = line.split('\t')
            if len(cols) != 6:
                continue
            audio, speaker, dialect, pinyin, hakka, mandarin = cols
            audio = audio.lstrip('\ufeff')
            if audio == '音訊檔案名稱':
                continue
            role  = 'doctor' if speaker.strip() == 'speaker01' else 'patient'
            hakka = hakka.strip()
            if hakka:
                raw.append((role, hakka))
        if not raw:
            continue
        # Merge consecutive same-role lines
        turns = []
        for role, text in raw:
            if turns and turns[-1][0] == role:
                turns[-1] = (role, turns[-1][1] + ' ' + text)
            else:
                turns.append([role, text])
        name  = os.path.basename(f).replace('.txt', '')
        topic = name.rstrip('0123456789')
        dialogues.append({
            'id'    : name,
            'topic' : topic,
            'source': 'TSV',
            'turns' : [(r, t) for r, t in turns],
        })
    return dialogues


qa_dialogues  = load_qa_dialogues(QA_DIR)
tsv_dialogues = load_tsv_dialogues(TSV_DIR)

print(f"客語漢字QA : {len(qa_dialogues)} dialogues")
print(f"HAKKA_TSV  : {len(tsv_dialogues)} dialogues")

# Sanity check
bad = [d for d in qa_dialogues + tsv_dialogues
       if d['turns'] and d['turns'][0][0] != 'doctor']
print(f"Dialogues NOT starting with doctor: {len(bad)}  (should be 0)")

# Show turn count
total_turns  = sum(len(d['turns']) for d in qa_dialogues)
total_rounds = sum(len(d['turns']) // 2 for d in qa_dialogues)
print(f"Total QA turns  : {total_turns}")
print(f"Total QA rounds : {total_rounds}")

# Sample
sample = qa_dialogues[0]
print(f"\nSample: {sample['id']}")
for role, text in sample['turns'][:4]:
    print(f"  [{role:7s}] {text[:70]}")

客語漢字QA : 41 dialogues
HAKKA_TSV  : 12 dialogues
Dialogues NOT starting with doctor: 5  (should be 0)
Total QA turns  : 828
Total QA rounds : 398

Sample: 全身性症狀1
  [doctor ] 恁會早，請坐。今晡日仰仔呢？
  [patient] 先生，𠊎對昨暗晡開始作燒，直直燒到今，大約有三十八點五度。
  [doctor ] 恁樣哦。除忒作燒以外，還有其他个症頭無？比將講喉嗹痛、嗽、流鼻、頭那痛？
  [patient] 喉嗹頭有一息仔痛，還過歸身仔痠痛，頭那乜有兜仔暈暈，毋過無嗽抑係流鼻。


In [3]:
from collections import Counter

# ── Dedup: TSV ⊆ QA (12 dialogues overlap) ────────────────────────────────────
# QA already contains the Hakka side of all TSV dialogues → use QA as canonical.
# Only add TSV dialogues whose id is NOT in QA.
qa_ids = {d['id'] for d in qa_dialogues}
extra  = [d for d in tsv_dialogues if d['id'] not in qa_ids]

all_dialogues = qa_dialogues + extra
random.shuffle(all_dialogues)

# ── Stats ─────────────────────────────────────────────────────────────────────
topic_counts = Counter(d['topic'] for d in all_dialogues)
total_turns  = sum(len(d['turns']) for d in all_dialogues)
total_rounds = sum(len(d['turns']) // 2 for d in all_dialogues)

print(f"Total dialogues : {len(all_dialogues)}")
print(f"Total turns     : {total_turns}")
print(f"Total rounds    : {total_rounds}")
print(f"\nBy topic:")
for topic, cnt in sorted(topic_counts.items()):
    print(f"  {topic:<12}: {cnt} dialogues")

Total dialogues : 41
Total turns     : 828
Total rounds    : 398

By topic:
  全身性症狀       : 10 dialogues
  腎臟泌尿系統      : 9 dialogues
  腹部腸胃肝膽      : 13 dialogues
  軀幹四肢關節      : 9 dialogues


## 4. Train / Test Split (by dialogue)

In [4]:
# ── Hold out 1 dialogue per topic (up to TEST_N total) ────────────────────────
topics = list(topic_counts.keys())
test_ids  = set()
test_pool = {t: [d for d in all_dialogues if d['topic'] == t] for t in topics}

for topic in sorted(test_pool):
    if len(test_ids) >= TEST_N:
        break
    candidates = test_pool[topic]
    if candidates:
        picked = random.choice(candidates)
        test_ids.add(picked['id'])

train_dialogues = [d for d in all_dialogues if d['id'] not in test_ids]
test_dialogues  = [d for d in all_dialogues if d['id'] in test_ids]

print(f"Train : {len(train_dialogues)} dialogues  "
      f"({sum(len(d['turns'])//2 for d in train_dialogues)} rounds)")
print(f"Test  : {len(test_dialogues)} dialogues  "
      f"({sum(len(d['turns'])//2 for d in test_dialogues)} rounds)")
print(f"\nTest dialogues held out:")
for d in test_dialogues:
    print(f"  {d['id']}  ({len(d['turns'])} turns)")

Train : 37 dialogues  (361 rounds)
Test  : 4 dialogues  (37 rounds)

Test dialogues held out:
  腹部腸胃肝膽9  (19 turns)
  軀幹四肢關節6  (21 turns)
  腎臟泌尿系統7  (17 turns)
  全身性症狀5  (20 turns)


## 5. Build Sliding Window Examples

In [5]:
def build_examples(dialogues, window_size=WINDOW_SIZE):
    """
    Sliding window: cho mỗi Doctor turn (không phải turn đầu tiên),
    lấy tối đa `window_size` turns trước đó làm context.
    Trả về list of {'context': [(role, text), ...], 'response': str}
    
    context luôn kết thúc bằng Patient turn (để model predict Doctor).
    """
    examples = []
    for d in dialogues:
        turns = d['turns']
        for i, (role, text) in enumerate(turns):
            if role != 'doctor':
                continue
            if i == 0:
                continue  # skip opening greeting (no patient context yet)
            # context = up to window_size turns before i
            start = max(0, i - window_size)
            ctx = turns[start:i]
            # ensure context ends with patient turn
            if not ctx or ctx[-1][0] != 'patient':
                continue
            # drop leading doctor turns so context starts with patient
            while ctx and ctx[0][0] == 'doctor':
                ctx = ctx[1:]
            if not ctx:
                continue
            examples.append({
                'context' : ctx,
                'response': text,
                'topic'   : d['topic'],
                'dial_id' : d['id'],
            })
    return examples


train_examples = build_examples(train_dialogues)
test_examples  = build_examples(test_dialogues)
random.shuffle(train_examples)

print(f"Train examples : {len(train_examples)}")
print(f"Test  examples : {len(test_examples)}")

# Show 1 sample
ex = train_examples[0]
print("\n=== Sample training example ===")
print(f"Topic : {ex['topic']}")
for role, text in ex['context']:
    label = '[Patient]' if role == 'patient' else '[Doctor ]'
    print(f"  {label} {text}")
print(f"  → [Doctor ] {ex['response']}")

Train examples : 353
Test  examples : 36

=== Sample training example ===
Topic : 腎臟泌尿系統
  [Patient] 𠊎有高血壓，有食降血壓个藥仔，其他無麼个病。
  [Doctor ] 脝水可能同心臟功能毋好、腎家功能無正常、肝病抑係藥仔个副作用有關係。
  [Patient] 𠊎會安排檢血、「腎功能檢查」、心臟同肚屎个超音波。
  [Doctor ] 脝水會當嚴重無？敢使愛戴院無？
  [Patient] 看檢查結果正來打算，早期發現過治療，一般控制到會做得。
  → [Doctor ] 係講有敨氣困難抑係歸身嚴重个脝水，就可能愛戴院。


## 6. Load Model (Unsloth)

In [8]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "./outputs/hakka_translation",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

# model = FastLanguageModel.get_peft_model(
#     model,
#     r                   = LORA_R,
#     lora_alpha          = LORA_ALPHA,
#     lora_dropout        = LORA_DROPOUT,
#     target_modules      = TARGET_MODULES,
#     bias                = "none",
#     use_gradient_checkpointing = "unsloth",
#     random_state        = RANDOM_SEED,
# )
print(model.print_trainable_parameters())

==((====))==  Unsloth 2026.4.5: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 2. Max memory: 23.527 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Unsloth 2026.4.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,220,672 || trainable%: 0.5196
None


In [10]:
from huggingface_hub import login

login(token="")
# Merge LoRA + upload
model.push_to_hub_merged(
    "ngocthien/hakka-translation-model",
    tokenizer,
    save_method = "merged_16bit",
)

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `ngocthien/hakka-translation-model`: 100%|██████████| 4/4 [00:40<00:00, 10.19s/it]


Successfully copied all 4 files from cache to `ngocthien/hakka-translation-model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [01:04<03:12, 64.05s/it]No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit:  50%|█████     | 2/4 [01:56<01:54, 57.04s/it]No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [02:56<00:58, 58.53s/it]No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.

Unsloth: Merge process complete. Saved to `/workspace/finetune-hakka/notebooks/ngocthien/hakka-translation-model`


In [11]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "ngocthien/hakka-translation-model",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

==((====))==  Unsloth 2026.4.5: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 2. Max memory: 23.527 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = TARGET_MODULES,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = RANDOM_SEED,
)

In [13]:
print(model.print_trainable_parameters())

trainable params: 41,943,040 || all params: 8,072,220,672 || trainable%: 0.5196
None


## 7. Format & Tokenize

In [14]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="llama-3")


def example_to_messages(ex):
    """
    Convert a sliding-window example to chat messages.
    context: [(role, text), ...] ending with patient turn
    response: doctor's next turn
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for role, text in ex['context']:
        if role == 'patient':
            messages.append({"role": "user",      "content": text})
        else:
            messages.append({"role": "assistant", "content": text})
    messages.append({"role": "assistant", "content": ex['response']})
    return messages


def format_examples(batch):
    texts = []
    for ex in batch['example']:
        msgs = example_to_messages(ex)
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}


train_ds = Dataset.from_list([{'example': ex} for ex in train_examples])
train_ds = train_ds.map(format_examples, batched=True)

print(f"Train dataset : {len(train_ds):,} examples")
print("\n=== Sample formatted text ===")
print(train_ds[0]['text'][:600], "...")

Map:   0%|          | 0/353 [00:00<?, ? examples/s]

Train dataset : 353 examples

=== Sample formatted text ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

你係一位客語（四縣腔）醫生，請用純客語（四縣腔）同病人進行醫療問診。毋好用華語抑係英文回覆，所有回覆愛用客語漢字。<|eot_id|><|start_header_id|>user<|end_header_id|>

𠊎有高血壓，有食降血壓个藥仔，其他無麼个病。<|eot_id|><|start_header_id|>assistant<|end_header_id|>

脝水可能同心臟功能毋好、腎家功能無正常、肝病抑係藥仔个副作用有關係。<|eot_id|><|start_header_id|>user<|end_header_id|>

𠊎會安排檢血、「腎功能檢查」、心臟同肚屎个超音波。<|eot_id|><|start_header_id|>assistant<|end_header_id|>

脝水會當嚴重無？敢使愛戴院無？<|eot_id|><|start_header_id|>user<|end_header_id|>

看檢查結果正來打算，早期發現過治療，一般控制到會做得。<|eot_id|><|start_header_id|>assistant<|end_header_id|>

係講有敨氣困難抑係歸身嚴重个脝水，就可能愛戴院。<|eot_id ...


## 8. Fine-tune (SFTTrainer)

**Format chuẩn theo Unsloth conversational template:**
- `DataCollatorForSeq2Seq` — proper padding + label masking
- `train_on_responses_only` — chỉ train trên Doctor (assistant) turns, mask Patient/System tokens
- `lora_dropout = 0` — Unsloth fast patching tối ưu QKV/O/MLP layers
- `per_device_train_batch_size = 2` — effective batch = 2 × 4 = 8
- `lr_scheduler_type = "linear"` — theo chuẩn Unsloth

In [15]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    data_collator      = DataCollatorForSeq2Seq(tokenizer=tokenizer),  # proper padding + label masking
    packing            = False,
    args = SFTConfig(
        per_device_train_batch_size   = BATCH_SIZE,      # 2 (was hardcoded 50)
        gradient_accumulation_steps   = GRAD_ACCUM,
        max_steps                     = MAX_STEPS,       # 300 (was hardcoded 100)
        warmup_steps                  = WARMUP_STEPS,    # 5
        learning_rate                 = LR,
        fp16                          = not torch.cuda.is_bf16_supported(),
        bf16                          = torch.cuda.is_bf16_supported(),
        logging_steps                 = 10,
        optim                         = "adamw_8bit",
        weight_decay                  = 0.01,
        lr_scheduler_type             = "linear",
        seed                          = RANDOM_SEED,
        output_dir                    = OUTPUT_DIR,
        report_to                     = "none",
    ),
)

# ── Only train on Doctor (assistant) responses — mask Patient/System tokens ───
# Without this, model would train on ALL tokens including patient questions,
# wasting capacity and learning contradictory patterns.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# ── Verify masking: labels for non-assistant tokens should be -100 ─────────────
space = tokenizer(" ", add_special_tokens=False).input_ids[0]
masked = tokenizer.decode(
    [space if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
).strip()
print("Masked labels sample (spaces = masked tokens):")
print(masked[:300], "...\n")

trainer_stats = trainer.train()
print(f"\nTraining done. Steps: {trainer_stats.global_step}  "
      f"Loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/353 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/353 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/353 [00:00<?, ? examples/s]

Masked labels sample (spaces = masked tokens):
脝水可能同心臟功能毋好、腎家功能無正常、肝病抑係藥仔个副作用有關係。<|eot_id|>                                      脝水會當嚴重無？敢使愛戴院無？<|eot_id|>                                 係講有敨氣困難抑係歸身嚴重个脝水，就可能愛戴院。<|eot_id|> ...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 353 | Num Epochs = 7 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,220,672 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.750600
20,1.525400
30,1.311500
40,1.106900
50,0.732300
60,0.368600
70,0.372600
80,0.312400
90,0.246200
100,0.095200



Training done. Steps: 300  Loss: 0.2764


## 9. Evaluation Functions

- **BLEU**: character-level n-gram (reused from translation notebook)
- **ROUGE-1/2/L**: character-level overlap
- **Perplexity (PPL)**: NLL chỉ trên Doctor response tokens (prompt tokens bị mask = -100)
  - Nhất quán với `train_on_responses_only` — đo đúng khả năng model generate doctor turns
  - Trước đây `labels=ids` tính PPL trên toàn sequence → kết quả không chính xác (PPL=7.15 đáng ngờ một phần vì lý do này)

In [16]:
# ── BLEU (reused from Hakka Translation notebook) ─────────────────────────────
import math
from collections import Counter
from typing import List, Sequence

def _norm(text): return "".join((text or "").strip().split())
def _tok(text):  return list(_norm(text))

def _ngrams(tokens: Sequence[str], n: int) -> Counter:
    if len(tokens) < n: return Counter()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def sentence_bleu(reference: str, hypothesis: str, max_n: int = 4, smooth: float = 1.0) -> float:
    ref, hyp = _tok(reference), _tok(hypothesis)
    if not hyp: return 0.0
    precs = []
    for n in range(1, max_n+1):
        rc, hc = _ngrams(ref, n), _ngrams(hyp, n)
        clip = total = 0
        for ng, cnt in hc.items():
            clip  += min(cnt, rc.get(ng, 0))
            total += cnt
        precs.append((clip+smooth)/(total+smooth) if total else 0.0)
    if min(precs) <= 0: return 0.0
    geo = math.exp(sum(math.log(p) for p in precs) / max_n)
    bp  = 1.0 if len(hyp) >= len(ref) else math.exp(1.0 - len(ref)/max(len(hyp),1))
    return bp * geo

def corpus_bleu(references: List[str], hypotheses: List[str], max_n: int = 4) -> float:
    if len(references) != len(hypotheses) or not references: return 0.0
    clip_tot = [0]*max_n; hyp_tot = [0]*max_n
    ref_len = hyp_len = 0
    for ref, hyp in zip(references, hypotheses):
        rt, ht = _tok(ref), _tok(hyp)
        ref_len += len(rt); hyp_len += len(ht)
        for n in range(1, max_n+1):
            rc, hc = _ngrams(rt, n), _ngrams(ht, n)
            for ng, cnt in hc.items():
                clip_tot[n-1] += min(cnt, rc.get(ng, 0))
            hyp_tot[n-1] += len(ht) - n + 1 if len(ht) >= n else 0
    precs = []
    for n in range(max_n):
        precs.append(clip_tot[n]/max(hyp_tot[n],1))
    if min(precs) <= 0: return 0.0
    geo = math.exp(sum(math.log(max(p,1e-9)) for p in precs) / max_n)
    bp  = 1.0 if hyp_len >= ref_len else math.exp(1.0 - ref_len/max(hyp_len,1))
    return bp * geo


# ── ROUGE (character-level) ───────────────────────────────────────────────────
def _lcs_len(x, y):
    """LCS length via DP."""
    m, n = len(x), len(y)
    if m == 0 or n == 0: return 0
    dp = [[0]*(n+1) for _ in range(2)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i%2][j] = dp[(i-1)%2][j-1]+1 if x[i-1]==y[j-1] else max(dp[(i-1)%2][j], dp[i%2][j-1])
    return dp[m%2][n]

def rouge_scores(reference: str, hypothesis: str) -> dict:
    """Compute ROUGE-1, ROUGE-2, ROUGE-L (character-level)."""
    ref, hyp = _tok(reference), _tok(hypothesis)
    if not ref or not hyp:
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    def f1(p, r): return 2*p*r/(p+r) if (p+r) > 0 else 0.0

    # ROUGE-1
    rc1, hc1 = Counter(ref), Counter(hyp)
    hit1 = sum(min(rc1[c], hc1[c]) for c in hc1)
    r1 = f1(hit1/len(hyp), hit1/len(ref))

    # ROUGE-2
    rc2, hc2 = _ngrams(ref, 2), _ngrams(hyp, 2)
    hit2 = sum(min(rc2[g], hc2[g]) for g in hc2)
    denom_r2 = max(len(hyp)-1, 1); denom_ref2 = max(len(ref)-1, 1)
    r2 = f1(hit2/denom_r2, hit2/denom_ref2)

    # ROUGE-L
    lcs = _lcs_len(ref, hyp)
    rL = f1(lcs/len(hyp), lcs/len(ref))

    return {'rouge1': r1, 'rouge2': r2, 'rougeL': rL}


# ── Generation helper ─────────────────────────────────────────────────────────
def generate_doctor_response(context, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS):
    """Given context [(role,text),...] ending with patient, generate Doctor reply."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for role, text in context:
        r = "user" if role == 'patient' else "assistant"
        messages.append({"role": r, "content": text})
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# ── Perplexity helper ─────────────────────────────────────────────────────────
def compute_ppl(examples, model, tokenizer):
    """
    Average PPL on the Doctor response tokens ONLY.

    Method: tokenize full sequence + prompt-only sequence, mask prompt tokens
    in labels (-100), so loss is computed only on the final doctor response.
    This is consistent with train_on_responses_only training objective.
    """
    model.eval()
    total_nll    = 0.0
    total_tokens = 0

    with torch.no_grad():
        for ex in examples:
            msgs = example_to_messages(ex)           # [..., {assistant: response}]
            prompt_msgs = msgs[:-1]                  # exclude final doctor turn

            full_text   = tokenizer.apply_chat_template(msgs,        tokenize=False, add_generation_prompt=False)
            prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)

            full_ids   = tokenizer(full_text,   return_tensors="pt")["input_ids"].to(model.device)
            prompt_ids = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(model.device)

            if full_ids.shape[1] > MAX_SEQ_LENGTH:
                continue

            prompt_len = prompt_ids.shape[1]
            resp_len   = full_ids.shape[1] - prompt_len
            if resp_len <= 0:
                continue

            # Mask prompt tokens — only compute loss on doctor response tokens
            labels = full_ids.clone()
            labels[0, :prompt_len] = -100

            out = model(full_ids, labels=labels)
            # out.loss is mean NLL over non-masked tokens
            total_nll    += out.loss.item() * resp_len
            total_tokens += resp_len

    ppl = math.exp(total_nll / max(total_tokens, 1))
    return round(ppl, 4)


print("Evaluation functions loaded.")

Evaluation functions loaded.


## 10. Baseline Evaluation (Zero-shot)

Đánh giá **base model chưa fine-tune** trên test set với cùng prompt.

In [19]:
from unsloth import FastLanguageModel as FLM

# Load base model (no LoRA, inference mode)
base_model, base_tokenizer = FLM.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="llama-3")
FLM.for_inference(base_model)
print("Base model loaded for inference.")

==((====))==  Unsloth 2026.4.5: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 2. Max memory: 23.527 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base model loaded for inference.


In [17]:
def run_eval(model, tok, examples, label):
    """
    Run BLEU + ROUGE + PPL evaluation.
    Returns summary dict + detailed rows.
    """
    print(f"\n{'─'*55}")
    print(f"Evaluating: {label}  ({len(examples)} examples)")
    print(f"{'─'*55}")

    refs_all, hyps_all = [], []
    rows = []
    bleus, r1s, r2s, rLs = [], [], [], []

    for i, ex in enumerate(examples):
        ref  = ex['response']
        pred = generate_doctor_response(ex['context'], model, tok)

        bleu = sentence_bleu(ref, pred)
        rg   = rouge_scores(ref, pred)

        bleus.append(bleu)
        r1s.append(rg['rouge1'])
        r2s.append(rg['rouge2'])
        rLs.append(rg['rougeL'])
        refs_all.append(ref)
        hyps_all.append(pred)
        rows.append({
            'topic': ex['topic'],
            'reference' : ref,
            'prediction': pred,
            'bleu' : round(bleu, 4),
            'rouge1': round(rg['rouge1'], 4),
            'rouge2': round(rg['rouge2'], 4),
            'rougeL': round(rg['rougeL'], 4),
        })
        if (i+1) % 10 == 0:
            print(f"  [{i+1}/{len(examples)}] avg BLEU={sum(bleus)/len(bleus):.4f}")

    c_bleu = corpus_bleu(refs_all, hyps_all)
    ppl    = compute_ppl(examples, model, tok)

    summary = {
        'label'        : label,
        'n'            : len(examples),
        'avg_bleu'     : round(sum(bleus)/len(bleus), 4),
        'corpus_bleu'  : round(c_bleu, 4),
        'avg_rouge1'   : round(sum(r1s)/len(r1s), 4),
        'avg_rouge2'   : round(sum(r2s)/len(r2s), 4),
        'avg_rougeL'   : round(sum(rLs)/len(rLs), 4),
        'perplexity'   : ppl,
    }
    print(f"\n  avg BLEU     = {summary['avg_bleu']:.4f}")
    print(f"  corpus BLEU  = {summary['corpus_bleu']:.4f}")
    print(f"  avg ROUGE-1  = {summary['avg_rouge1']:.4f}")
    print(f"  avg ROUGE-2  = {summary['avg_rouge2']:.4f}")
    print(f"  avg ROUGE-L  = {summary['avg_rougeL']:.4f}")
    print(f"  Perplexity   = {summary['perplexity']:.2f}")
    return summary, rows


In [20]:
baseline_summary, baseline_rows = run_eval(
    base_model, base_tokenizer, test_examples, label="Baseline (zero-shot)"
)


───────────────────────────────────────────────────────
Evaluating: Baseline (zero-shot)  (36 examples)
───────────────────────────────────────────────────────
  [10/36] avg BLEU=0.0400
  [20/36] avg BLEU=0.0508
  [30/36] avg BLEU=0.0547

  avg BLEU     = 0.0542
  corpus BLEU  = 0.0257
  avg ROUGE-1  = 0.2206
  avg ROUGE-2  = 0.0462
  avg ROUGE-L  = 0.1753
  Perplexity   = 43.73


## 11. Fine-tuned Model Evaluation

In [18]:
FastLanguageModel.for_inference(model)  # enable fast inference mode

ft_summary, ft_rows = run_eval(
    model, tokenizer, test_examples, label="Fine-tuned (LoRA)"
)


───────────────────────────────────────────────────────
Evaluating: Fine-tuned (LoRA)  (36 examples)
───────────────────────────────────────────────────────
  [10/36] avg BLEU=0.1906
  [20/36] avg BLEU=0.1397
  [30/36] avg BLEU=0.1451

  avg BLEU     = 0.1387
  corpus BLEU  = 0.1113
  avg ROUGE-1  = 0.3208
  avg ROUGE-2  = 0.1464
  avg ROUGE-L  = 0.2624
  Perplexity   = 9.04


## 12. Comparison Summary

In [21]:
def delta(a, b, higher_better=True):
    diff = b - a
    sign = '+' if diff >= 0 else ''
    arrow = '▲' if (diff > 0) == higher_better else ('▼' if diff != 0 else '─')
    return f"{sign}{diff:.4f} {arrow}"

bs, ft = baseline_summary, ft_summary

print("\n" + "═"*72)
print(f"{'Metric':<18} {'Baseline':>12} {'Fine-tuned':>12} {'Delta':>18}")
print("─"*72)
metrics = [
    ('avg BLEU',    'avg_bleu',    True),
    ('corpus BLEU', 'corpus_bleu', True),
    ('ROUGE-1',     'avg_rouge1',  True),
    ('ROUGE-2',     'avg_rouge2',  True),
    ('ROUGE-L',     'avg_rougeL',  True),
    ('Perplexity',  'perplexity',  False),  # lower = better
]
for name, key, hb in metrics:
    bv = bs[key]; fv = ft[key]
    print(f"{name:<18} {bv:>12.4f} {fv:>12.4f} {delta(bv, fv, hb):>18}")
print("═"*72)
print(f"Test examples  : {ft['n']}")
print(f"Model          : {MODEL_ID}")
print(f"Window size    : {WINDOW_SIZE} turns")


════════════════════════════════════════════════════════════════════════
Metric                 Baseline   Fine-tuned              Delta
────────────────────────────────────────────────────────────────────────
avg BLEU                 0.0542       0.1387          +0.0845 ▲
corpus BLEU              0.0257       0.1113          +0.0856 ▲
ROUGE-1                  0.2206       0.3208          +0.1002 ▲
ROUGE-2                  0.0462       0.1464          +0.1002 ▲
ROUGE-L                  0.1753       0.2624          +0.0871 ▲
Perplexity              43.7322       9.0421         -34.6901 ▲
════════════════════════════════════════════════════════════════════════
Test examples  : 36
Model          : yentinglin/Llama-3-Taiwan-8B-Instruct
Window size    : 6 turns


In [22]:
# ── Qualitative comparison: same context, both models ─────────────────────────
print("=== Qualitative comparison (5 examples) ===")
for i, ex in enumerate(test_examples[:5]):
    print(f"\n[{i+1}] Topic: {ex['topic']}")
    print("Context:")
    for role, text in ex['context'][-2:]:  # show last 2 turns
        label = 'Patient' if role == 'patient' else 'Doctor '
        print(f"  [{label}] {text}")
    print(f"Reference : {ex['response']}")
    print(f"Baseline  : {baseline_rows[i]['prediction']}")
    print(f"Fine-tuned: {ft_rows[i]['prediction']}")
    print(f"BLEU  baseline={baseline_rows[i]['bleu']:.4f}  ft={ft_rows[i]['bleu']:.4f}")

=== Qualitative comparison (5 examples) ===

[1] Topic: 腹部腸胃肝膽
Context:
  [Patient] 先生，𠊎這幾日屙屎个時節有發現屙个屎有異多血，看起來有兜仔嚴重，故所遽遽來檢查。
Reference : 恁樣哦。請問你看著个血係麼个色？正紅色？烏𪐞紅？抑係像馬膠共樣烏疏疏仔？
Baseline  : 你好！我明白你最近屙屎時有發現血，擔心有兜仔嚴重，故此來檢查。請問你有無其他症狀，例如腹痛、腹瀉、食慾不振等？你最近有無特別緊張或操勞？請詳細告訴我你的症狀，我會盡力幫你診斷問題。
Fine-tuned: 𠊎知哩。請問你屙个屎係有血、黏在屎胐面，抑係屙个時節屎胐空該搭仔血呢？
BLEU  baseline=0.0428  ft=0.1252

[2] Topic: 腹部腸胃肝膽
Context:
  [Doctor ] 恁樣哦。請問你看著个血係麼个色？正紅色？烏𪐞紅？抑係像馬膠共樣烏疏疏仔？
  [Patient] 係烏𪐞紅色个，有成時混雜在屎肚，乜有時節係在屙忒過後有看著滴血。
Reference : 大約係對哪央時開始个啊？每日都有嘎？
Baseline  : 你講个係烏𪐞紅色个血，乜有時節係在屙忒過後有看著滴血。這可能係腸胃出現問題，例如潰瘍或發炎。請問你有無其他症狀，例如腹痛、腹瀉、噁心或嘔吐？
Fine-tuned: 敢有發燒、肚屎痛、畏胲抑係想翻个情形？
BLEU  baseline=0.0225  ft=0.0810

[3] Topic: 腹部腸胃肝膽
Context:
  [Doctor ] 大約係對哪央時開始个啊？每日都有嘎？
  [Patient] 差毋多四日前開始个，怕係逐日都有，還兼屙屎過後肚屎會陰痛陰痛仔。
Reference : 除忒屙屎有血以外，敢有發燒、翻、屙痢肚、人變較瘦抑係試著𤸁𤸁个情況？
Baseline  : 你有無發現其他症狀？比如說，排尿有無疼痛、腹部有無脹氣、食慾有無改變等等。
Fine-tuned: 敢有屙屎時節會看著个「腸仔」發炎？抑係屙屎過後會痛無啊？
BLEU  baseline=0.0475  ft=0.0773

[4] Topic: 腹部腸胃肝膽
Context:
  [Doctor ] 除忒屙屎有血以外，敢有發燒、翻、屙痢肚、人變較瘦抑係

## 13. Save Model

In [23]:
# ── Save LoRA adapters ────────────────────────────────────────────────────────
SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapters")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapters saved → {SAVE_PATH}")

# ── Optional: merge & save full model ────────────────────────────────────────
# merged = model.merge_and_unload()
# merged.save_pretrained(os.path.join(OUTPUT_DIR, "merged"))
# tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "merged"))

# ── Optional: push to HuggingFace Hub ────────────────────────────────────────
# model.push_to_hub("your-username/hakka-dialogue-doctor", token="hf_...")

LoRA adapters saved → outputs/hakka_dialogue/lora_adapters


## 14. Inference Demo

Thử chat với model — nhập câu của bệnh nhân bằng tiếng Hakka.

In [24]:
def chat_demo(patient_turns, model, tokenizer):
    """
    patient_turns: list of patient Hakka utterances (simulating conversation history)
    Returns doctor's response.
    """
    context = [('patient', t) for t in patient_turns]
    resp = generate_doctor_response(context, model, tokenizer)
    return resp


# Example 1: 腎臟泌尿系統 (kidney/urinary)
history_1 = [
    "先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。"
]
print("=== Demo 1: 泌尿 ===" )
print(f"Patient : {history_1[-1]}")
print(f"Doctor  : {chat_demo(history_1, model, tokenizer)}")

print()

# Example 2: 全身性症狀 (systemic)
history_2 = [
    "先生，𠊎對昨暗晡開始作燒，直直燒到今，大約有三十八點五度。"
]
print("=== Demo 2: 發燒 ===")
print(f"Patient : {history_2[-1]}")
print(f"Doctor  : {chat_demo(history_2, model, tokenizer)}")

print()

# Example 3: 頭痛 (headache)
history_3 = [
    "先生，𠊎頭那目珠直直漲，頭殼痛了好幾日，食藥仔也毋見好。"
]
print("=== Demo 3: 頭痛 ===")
print(f"Patient : {history_3[-1]}")
print(f"Doctor  : {chat_demo(history_3, model, tokenizer)}")

print()

# Example 4: 腸胃 (gastrointestinal)
history_4 = [
    "先生，𠊎肚子痛毋舒服，還有想愛嘔，食野就嘔，已經兩工了。"
]
print("=== Demo 4: 腸胃 ===")
print(f"Patient : {history_4[-1]}")
print(f"Doctor  : {chat_demo(history_4, model, tokenizer)}")

print()

# Example 5: 呼吸道 (respiratory)
history_5 = [
    "先生，𠊎這駁仔一直在咳，喉嚨那燒燒忒，有時還流鼻水，覺得無力。"
]
print("=== Demo 5: 呼吸道 ===")
print(f"Patient : {history_5[-1]}")
print(f"Doctor  : {chat_demo(history_5, model, tokenizer)}")

print()

# Example 6: 失眠 (insomnia)
history_6 = [
    "先生，𠊎這陣子夜夜覺毋著，躺等床項想東想西，眠毋到兩三點，日時頭精神恁差。"
]
print("=== Demo 6: 失眠 ===")
print(f"Patient : {history_6[-1]}")

=== Demo 1: 泌尿 ===
Patient : 先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。
Doctor  : 恁樣哦。請問恁仔續等有幾久哩？除忒密密愛屙尿以外，敢會尿出、屙尿痛抑係半夜仔愛䟘起來屙尿个情形？

=== Demo 2: 發燒 ===
Patient : 先生，𠊎對昨暗晡開始作燒，直直燒到今，大約有三十八點五度。
Doctor  : 恁樣哦。除忒作燒以外，還有其他个症頭無？比將講喉嗹痛、嗽、流鼻、頭那痛？

=== Demo 3: 頭痛 ===
Patient : 先生，𠊎頭那目珠直直漲，頭殼痛了好幾日，食藥仔也毋見好。
Doctor  : 恁樣哦。請問你係對麼个時節開始个？頭那位所痛？有遰對其他位所去無？

=== Demo 4: 腸胃 ===
Patient : 先生，𠊎肚子痛毋舒服，還有想愛嘔，食野就嘔，已經兩工了。
Doctor  : 恁樣哦。請問你肚屎痛有幾久仔哩？肚屎个哪位痛？有遰對其他位所去無？

=== Demo 5: 呼吸道 ===
Patient : 先生，𠊎這駁仔一直在咳，喉嚨那燒燒忒，有時還流鼻水，覺得無力。
Doctor  : 這種情形差毋多續等有幾久吔？

=== Demo 6: 失眠 ===
Patient : 先生，𠊎這陣子夜夜覺毋著，躺等床項想東想西，眠毋到兩三點，日時頭精神恁差。


## 15. Multi-turn Conversation Demo

Mô phỏng cuộc hội thoại **liên tục nhiều lượt** — bệnh nhân trả lời câu hỏi bác sĩ, bác sĩ tiếp tục hỏi thêm cho đến khi đưa ra chẩn đoán/hướng điều trị.

In [25]:
def multi_turn_chat(patient_script, model, tokenizer, topic=""):
    """
    Simulate a full multi-turn doctor–patient dialogue.

    patient_script : list of pre-written patient utterances (one per round).
                     The model generates the doctor reply after each one.
    history        : [(role, text), ...] grows turn by turn.
    """
    history = []
    sep = "─" * 60

    if topic:
        print(f"\n{'═'*60}")
        print(f"  {topic}")
        print(f"{'═'*60}")

    for turn_idx, patient_text in enumerate(patient_script, start=1):
        # ── Patient speaks ────────────────────────────────────────
        history.append(('patient', patient_text))
        print(f"\n[Turn {turn_idx}]")
        print(f"  Patient : {patient_text}")

        # ── Doctor replies (model generates with full history) ────
        doctor_text = generate_doctor_response(history, model, tokenizer)
        history.append(('doctor', doctor_text))
        print(f"  Doctor  : {doctor_text}")
        print(f"  {sep}")

    return history


# ══════════════════════════════════════════════════════════════════
# Conversation 1: 泌尿道感染 (UTI) — 4 rounds
# ══════════════════════════════════════════════════════════════════
script_uti = [
    # Round 1 — chief complaint
    "先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。",
    # Round 2 — answer doctor's follow-up (duration + dysuria + nocturia)
    "已經三四工了，屙尿个時節有淡薄仔刺刺个感覺，半夜仔也愛起來兩三趟。",
    # Round 3 — answer about fever / urine colour
    "𠊎有作燒，昨日量著三十七點八度，尿也有淡薄仔混濁，顏色較深。",
    # Round 4 — medical history / allergies
    "𠊎無過敏史，毋過上個月有食過幾日止痛藥。",
]

hist_uti = multi_turn_chat(script_uti, model, tokenizer,
                           topic="Conversation 1 : 泌尿道感染 (UTI)")

print()

# ══════════════════════════════════════════════════════════════════
# Conversation 2: 上呼吸道感染 (Upper respiratory) — 4 rounds
# ══════════════════════════════════════════════════════════════════
script_uri = [
    # Round 1 — chief complaint
    "先生，𠊎這駁仔一直在咳，喉嚨那燒燒忒，有時還流鼻水，覺得無力。",
    # Round 2 — duration + fever + sore throat
    "大約四五工了，昨暗晡量著三十八度，喉嚨有當痛，吞食野也痛。",
    # Round 3 — cough characteristics
    "咳出來有淡薄仔痰，係黃色个，胸肚項有時會悶悶忒个感覺。",
    # Round 4 — contact history / allergy
    "𠊎上個禮拜有去人多个地方，屋家人也有人感冒，𠊎毋係過敏體質。",
]

hist_uri = multi_turn_chat(script_uri, model, tokenizer,
                           topic="Conversation 2 : 上呼吸道感染 (URI)")

print()

# ══════════════════════════════════════════════════════════════════
# Conversation 3: 腸胃炎 (Gastroenteritis) — 4 rounds
# ══════════════════════════════════════════════════════════════════
script_gi = [
    # Round 1 — chief complaint
    "先生，𠊎肚子痛毋舒服，還有想愛嘔，食野就嘔，已經兩工了。",
    # Round 2 — diarrhoea + fever
    "有屙肚，一工大概三四趟，有淡薄仔作燒，大約三十七點五度。",
    # Round 3 — food history
    "前日𠊎有去食海鮮，食完後差毋多兩點鐘就開始毋舒服了。",
    # Round 4 — current condition
    "今仔日感覺有淡薄仔好轉，毋過肚子還係軟弱無力，毋敢食太多。",
]

hist_gi = multi_turn_chat(script_gi, model, tokenizer,
                          topic="Conversation 3 : 腸胃炎 (Gastroenteritis)")


════════════════════════════════════════════════════════════
  Conversation 1 : 泌尿道感染 (UTI)
════════════════════════════════════════════════════════════

[Turn 1]
  Patient : 先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。
  Doctor  : 恁樣哦。請問恁仔續等有幾久哩？除忒密密愛屙尿以外，敢會尿出、屙尿痛抑係半夜仔愛䟘起來屙尿个情形？
  ────────────────────────────────────────────────────────────

[Turn 2]
  Patient : 已經三四工了，屙尿个時節有淡薄仔刺刺个感覺，半夜仔也愛起來兩三趟。
  Doctor  : 你這駁仔食滾水有較多無？抑係有啉含咖啡因、酒精个飲料？
  ────────────────────────────────────────────────────────────

[Turn 3]
  Patient : 𠊎有作燒，昨日量著三十七點八度，尿也有淡薄仔混濁，顏色較深。
  Doctor  : 這可能係「尿路感染」、「尿泡過動症」抑係攝護腺个問題來牽帶个。𠊎會摎你安排尿水个分析同培養，確定係麼个情形。
  ────────────────────────────────────────────────────────────

[Turn 4]
  Patient : 𠊎無過敏史，毋過上個月有食過幾日止痛藥。
  Doctor  : 好，𠊎會先摎你磅尿、血，安排檢尿，係講有需要正來安排「肚屎超音波」抑係「尿流動力學檢查」。
  ────────────────────────────────────────────────────────────


════════════════════════════════════════════════════════════
  Conversation 2 : 上呼吸道感染 (URI)
════════════════════════════════════════════════════════════

[Turn 